# Extracting CropScape County-level Crop Acreage Data
The following code pulls County-level crop data estimates using each county's FIPS code and used the US Census Gazetteer files (https://www.census.gov/geographies/reference-files/time-series/geo/gazetteer-files.html) to map FIPS code to county name.

### Imports

In [1]:
import pandas as pd
import requests
import re
import os
from io import StringIO
from time import sleep

### Helper functions

In [2]:
BASE_URL = "https://nassgeodata.gmu.edu/axis2/services/CDLService/GetCDLStat"

NON_CONUS_STATE_FIPS = {"02", "15", "60", "66", "69", "72", "78"}

RAW_DIR = "../raw-files"
os.makedirs(RAW_DIR, exist_ok=True)

def load_counties_for_year(gazetteer_path):
    counties = pd.read_csv(
        gazetteer_path,
        sep="\t",
        dtype=str,
        encoding="latin1"
    )
    
    counties["county_fips"] = counties["GEOID"].astype(str).str.zfill(5)
    
    counties = counties.loc[
        ~counties["county_fips"].str[:2].isin(NON_CONUS_STATE_FIPS)
    ].copy()
    
    return counties

def fetch_one_county_cdl(year, fips, timeout=120):
    params = {
        "year": year,
        "fips": str(fips).zfill(5),
        "format": "csv"
    }

    r = requests.get(BASE_URL, params=params, timeout=timeout)
    r.raise_for_status()

    match = re.search(r"<returnURL>(.*?)</returnURL>", r.text)
    if not match:
        raise ValueError(f"Could not find returnURL for FIPS {fips}")

    csv_url = match.group(1)

    csv_response = requests.get(csv_url, timeout=timeout)
    csv_response.raise_for_status()

    df = pd.read_csv(StringIO(csv_response.text))
    df["county_fips"] = str(fips).zfill(5)
    df["year"] = year

    return df

### Main yearly runner with one retry pass

In [3]:
def run_cdl_year_with_retry(year, gazetteer_filename, checkpoint_every=50, sleep_sec=0.2):
    gazetteer_path = os.path.join(RAW_DIR, gazetteer_filename)
    progress_file = os.path.join(RAW_DIR, f"cdl_{year}_progress.csv")
    failed_file = os.path.join(RAW_DIR, f"cdl_{year}_failed.csv")

    counties = load_counties_for_year(gazetteer_path)
    county_fips_list = counties["county_fips"].astype(str).tolist()

    print(f"{year}: total CONUS counties in Gazetteer file = {len(county_fips_list)}")

    if os.path.exists(progress_file):
        existing_progress = pd.read_csv(progress_file, dtype={"county_fips": str})
        existing_progress["county_fips"] = existing_progress["county_fips"].astype(str).str.zfill(5)
        done_fips = set(existing_progress["county_fips"].unique())
        print(f"{year}: existing completed counties found = {len(done_fips)}")
    else:
        done_fips = set()
        print(f"{year}: no prior progress file found; starting fresh")

    remaining_fips = [f for f in county_fips_list if f not in done_fips]
    print(f"{year}: remaining counties to download = {len(remaining_fips)}")

    new_failed = []

    # First pass
    for i, fips in enumerate(remaining_fips, start=1):
        try:
            df = fetch_one_county_cdl(year, fips)

            df.to_csv(
                progress_file,
                mode="a",
                header=not os.path.exists(progress_file),
                index=False
            )

        except Exception as e:
            new_failed.append({
                "county_fips": str(fips).zfill(5),
                "error": str(e)
            })
            print(f"{year} FAILED first pass: {fips} -> {e}")

        if i % checkpoint_every == 0:
            pd.DataFrame(new_failed).to_csv(failed_file, index=False)
            print(f"{year}: first-pass checkpoint saved at {i} counties")

        sleep(sleep_sec)

    pd.DataFrame(new_failed).to_csv(failed_file, index=False)

    print(f"{year}: first pass complete")
    print(f"{year}: failed counties after first pass = {len(new_failed)}")

    # Retry pass
    retry_failed = []

    if len(new_failed) > 0:
        print(f"{year}: starting single retry pass")

        for j, row in enumerate(new_failed, start=1):
            fips = str(row["county_fips"]).zfill(5)

            try:
                df = fetch_one_county_cdl(year, fips)

                df.to_csv(
                    progress_file,
                    mode="a",
                    header=False,
                    index=False
                )

            except Exception as e:
                retry_failed.append({
                    "county_fips": fips,
                    "error": str(e)
                })
                print(f"{year} FAILED retry: {fips} -> {e}")

            if j % checkpoint_every == 0:
                pd.DataFrame(retry_failed).to_csv(failed_file, index=False)
                print(f"{year}: retry checkpoint saved at {j} failed counties processed")

            sleep(sleep_sec)

    pd.DataFrame(retry_failed).to_csv(failed_file, index=False)

    final_progress = pd.read_csv(progress_file, dtype={"county_fips": str})
    final_progress["county_fips"] = final_progress["county_fips"].astype(str).str.zfill(5)

    print(f"{year}: final raw rows saved = {final_progress.shape[0]}")
    print(f"{year}: final unique counties saved = {final_progress['county_fips'].nunique()}")
    print(f"{year}: counties still failed after retry = {len(retry_failed)}")

### 2017 data

In [4]:
run_cdl_year_with_retry(
    year=2017,
    gazetteer_filename="2017_counties_national.txt",
    checkpoint_every=50,
    sleep_sec=0.2
)

2017: total CONUS counties in Gazetteer file = 3108
2017: no prior progress file found; starting fresh
2017: remaining counties to download = 3108
2017: first-pass checkpoint saved at 50 counties
2017: first-pass checkpoint saved at 100 counties
2017: first-pass checkpoint saved at 150 counties
2017: first-pass checkpoint saved at 200 counties
2017: first-pass checkpoint saved at 250 counties
2017: first-pass checkpoint saved at 300 counties
2017: first-pass checkpoint saved at 350 counties
2017: first-pass checkpoint saved at 400 counties
2017: first-pass checkpoint saved at 450 counties
2017: first-pass checkpoint saved at 500 counties
2017: first-pass checkpoint saved at 550 counties
2017: first-pass checkpoint saved at 600 counties
2017: first-pass checkpoint saved at 650 counties
2017: first-pass checkpoint saved at 700 counties
2017: first-pass checkpoint saved at 750 counties
2017: first-pass checkpoint saved at 800 counties
2017: first-pass checkpoint saved at 850 counties
2017

### Quick validation

In [8]:
from pandas.errors import EmptyDataError

for year in [2017]:
    progress_file = os.path.join(RAW_DIR, f"cdl_{year}_progress.csv")
    failed_file = os.path.join(RAW_DIR, f"cdl_{year}_failed.csv")

    df = pd.read_csv(progress_file, dtype={"county_fips": str})
    print(year, "rows:", df.shape[0], "unique counties:", df["county_fips"].nunique())

    try:
        failed = pd.read_csv(failed_file, dtype={"county_fips": str})
        print(year, "failed after retry:", failed.shape[0])
    except (FileNotFoundError, EmptyDataError):
        print(year, "failed after retry: 0")

2017 rows: 96081 unique counties: 3108
2017 failed after retry: 0


### 2018 data

In [9]:
run_cdl_year_with_retry(
    year=2018,
    gazetteer_filename="2018_counties_national.txt",
    checkpoint_every=50,
    sleep_sec=0.2
)

2018: total CONUS counties in Gazetteer file = 3108
2018: no prior progress file found; starting fresh
2018: remaining counties to download = 3108
2018: first-pass checkpoint saved at 50 counties
2018: first-pass checkpoint saved at 100 counties
2018: first-pass checkpoint saved at 150 counties
2018: first-pass checkpoint saved at 200 counties
2018: first-pass checkpoint saved at 250 counties
2018: first-pass checkpoint saved at 300 counties
2018: first-pass checkpoint saved at 350 counties
2018: first-pass checkpoint saved at 400 counties
2018: first-pass checkpoint saved at 450 counties
2018: first-pass checkpoint saved at 500 counties
2018: first-pass checkpoint saved at 550 counties
2018: first-pass checkpoint saved at 600 counties
2018: first-pass checkpoint saved at 650 counties
2018: first-pass checkpoint saved at 700 counties
2018: first-pass checkpoint saved at 750 counties
2018: first-pass checkpoint saved at 800 counties
2018: first-pass checkpoint saved at 850 counties
2018

### Quick Validation

In [10]:
for year in [2018]:
    progress_file = os.path.join(RAW_DIR, f"cdl_{year}_progress.csv")
    failed_file = os.path.join(RAW_DIR, f"cdl_{year}_failed.csv")

    df = pd.read_csv(progress_file, dtype={"county_fips": str})
    print(year, "rows:", df.shape[0], "unique counties:", df["county_fips"].nunique())

    try:
        failed = pd.read_csv(failed_file, dtype={"county_fips": str})
        print(year, "failed after retry:", failed.shape[0])
    except (FileNotFoundError, EmptyDataError):
        print(year, "failed after retry: 0")

2018 rows: 107780 unique counties: 3107
2018 failed after retry: 0


### 2019 data

In [11]:
run_cdl_year_with_retry(
    year=2019,
    gazetteer_filename="2019_counties_national.txt",
    checkpoint_every=50,
    sleep_sec=0.2
)

2019: total CONUS counties in Gazetteer file = 3108
2019: no prior progress file found; starting fresh
2019: remaining counties to download = 3108
2019: first-pass checkpoint saved at 50 counties
2019: first-pass checkpoint saved at 100 counties
2019: first-pass checkpoint saved at 150 counties
2019: first-pass checkpoint saved at 200 counties
2019: first-pass checkpoint saved at 250 counties
2019: first-pass checkpoint saved at 300 counties
2019: first-pass checkpoint saved at 350 counties
2019: first-pass checkpoint saved at 400 counties
2019: first-pass checkpoint saved at 450 counties
2019: first-pass checkpoint saved at 500 counties
2019: first-pass checkpoint saved at 550 counties
2019: first-pass checkpoint saved at 600 counties
2019: first-pass checkpoint saved at 650 counties
2019: first-pass checkpoint saved at 700 counties
2019: first-pass checkpoint saved at 750 counties
2019: first-pass checkpoint saved at 800 counties
2019: first-pass checkpoint saved at 850 counties
2019

### Quick validation

In [12]:
for year in [2019]:
    progress_file = os.path.join(RAW_DIR, f"cdl_{year}_progress.csv")
    failed_file = os.path.join(RAW_DIR, f"cdl_{year}_failed.csv")

    df = pd.read_csv(progress_file, dtype={"county_fips": str})
    print(year, "rows:", df.shape[0], "unique counties:", df["county_fips"].nunique())

    try:
        failed = pd.read_csv(failed_file, dtype={"county_fips": str})
        print(year, "failed after retry:", failed.shape[0])
    except (FileNotFoundError, EmptyDataError):
        print(year, "failed after retry: 0")

2019 rows: 113579 unique counties: 3108
2019 failed after retry: 0


### Merge all Crop Data for 2016-2019

In [13]:
# combine all yearly raw CDL files
cdl_2016 = pd.read_csv("../raw-files/cdl_raw_2016_complete.csv", dtype={"county_fips": str})
cdl_2017 = pd.read_csv("../raw-files/cdl_2017_progress.csv", dtype={"county_fips": str})
cdl_2018 = pd.read_csv("../raw-files/cdl_2018_progress.csv", dtype={"county_fips": str})
cdl_2019 = pd.read_csv("../raw-files/cdl_2019_progress.csv", dtype={"county_fips": str})

cdl_raw_all = pd.concat([cdl_2016, cdl_2017, cdl_2018, cdl_2019], ignore_index=True)
cdl_raw_all["county_fips"] = cdl_raw_all["county_fips"].astype(str).str.zfill(5)

print(cdl_raw_all.shape)
print(cdl_raw_all["year"].value_counts().sort_index())

(425520, 6)
year
2016    108080
2017     96081
2018    107780
2019    113579
Name: count, dtype: int64


In [ ]:
# combine all yearly raw CDL files
cdl_2016 = pd.read_csv("../raw-files/cdl_raw_2016_complete.csv", dtype={"county_fips": str})
cdl_2017 = pd.read_csv("../raw-files/cdl_2017_progress.csv", dtype={"county_fips": str})
cdl_2018 = pd.read_csv("../raw-files/cdl_2018_progress.csv", dtype={"county_fips": str})
cdl_2019 = pd.read_csv("../raw-files/cdl_2019_progress.csv", dtype={"county_fips": str})

cdl_raw_all = pd.concat([cdl_2016, cdl_2017, cdl_2018, cdl_2019], ignore_index=True)
cdl_raw_all["county_fips"] = cdl_raw_all["county_fips"].astype(str).str.zfill(5)

print(cdl_raw_all.shape)
print(cdl_raw_all["year"].value_counts().sort_index())

(425520, 6)
year
2016    108080
2017     96081
2018    107780
2019    113579
Name: count, dtype: int64


### Load Census file linking FIPS codes to county names

In [16]:
# load one Gazetteer file for county names/state info
gazetteer = pd.read_csv(
    "../raw-files/2019_counties_national.txt",
    sep="\t",
    dtype=str,
    encoding="latin1"
)

gazetteer["county_fips"] = gazetteer["GEOID"].astype(str).str.zfill(5)

gazetteer = gazetteer.rename(columns={
    "NAME": "county_name",
    "USPS": "state_abbrev"
})

gazetteer = gazetteer[["county_fips", "county_name", "state_abbrev"]].drop_duplicates()

print(gazetteer.shape)
gazetteer.head()

(3220, 3)


,county_fips,county_name,state_abbrev
0,01001,Autauga County,AL
1,01003,Baldwin County,AL
2,01005,Barbour County,AL
3,01007,Bibb County,AL
4,01009,Blount County,AL


### Merge crop data with census file

In [17]:
# merge Gazetteer metadata onto combined raw CDL
cdl_raw_all = cdl_raw_all.merge(
    gazetteer,
    on="county_fips",
    how="left"
)

print(cdl_raw_all.shape)
cdl_raw_all[["county_fips", "county_name", "state_abbrev", "year"]].head()

(425520, 8)


,county_fips,county_name,state_abbrev,year
0,01001,Autauga County,AL,2016
1,01001,Autauga County,AL,2016
2,01001,Autauga County,AL,2016
3,01001,Autauga County,AL,2016
4,01001,Autauga County,AL,2016


### Save file

In [21]:
# save final crop acreage data per county with county names
cdl_raw_all.to_csv("../raw-files/cropdata_raw_2016_2019_county_names.csv", index=False)